In [1]:
import pandas as pd 

# Load combined cleaned pathology data frame
df_path = "D:\DATA\df_cleaned.xlsx"
df_all = pd.read_excel(df_path)

In [2]:
# Remove rendundant columns
cols_to_remove = ["mattype tekst", "makrotekst", "mikrotekst", "snomed kode", "kode fritekst","wsi count"]
df_selected = df_all.drop(columns = cols_to_remove)

print(df_selected.head())

     rekvnr   modtdato team sex  alder alder gruppe   rekvdato  matantal  \
0  12100029 2012-01-02  LUG   F     87          65+ 2012-01-02         3   
1  12102211 2012-02-09  LUG   F     36        35-39 2012-02-07         1   
2  12105184 2012-03-28  LUG   M     87          65+ 2012-03-27         1   
3  12200001 2012-01-02  URO   M     52        50-54 2011-12-30        12   
4  12200014 2012-01-02  LUG   F     87          65+ 2012-01-02         1   

   mattype                                      wsi filenames  \
0       24  ['//regsj.intern/appl/Deep_Visual_Proteomics\\...   
1       23  ['//regsj.intern/appl/Deep_Visual_Proteomics\\...   
2       26  ['//regsj.intern/appl/Deep_Visual_Proteomics\\...   
3       11  ['//regsj.intern/appl/Deep_Visual_Proteomics\\...   
4       11  ['//regsj.intern/appl/Deep_Visual_Proteomics\\...   

                                T                     M                 Other  
0  ('T2Y414', 'T08B07', 'T08B44')  ('M09462', 'M81406')  ('P31067', 'ÆF4

In [3]:
import ast

def convert_string_to_list(df, columns):
    for col in columns:
        df[col] = df[col].apply(lambda x: ast.literal_eval(x) if pd.notnull(x) else x)
    return df

df_selected = convert_string_to_list(df_selected, ["wsi filenames", "T", "M", "Other"])

In [4]:
from snomed_hierarchy import SNOMEDCodes, SNOMEDHierarchy

# Load SNOMED codes
snomed_path = "D:/DATA/patoSnoMed_2025-04.xlsx"
xls_snomed = pd.read_excel(snomed_path)
df_snomed = pd.DataFrame(xls_snomed, columns=['SKSkode', 'DatoFra', 'DatoÆndring', 'DatoTil', 'Kodetekst', 'Fuldtekst'])
snomed = SNOMEDCodes(df_snomed)

In [5]:
# Create morphology hierarchy

# Get df of M codes
m_codes = snomed.get_codes_by_letter('M')

all_m_codes = df_selected["M"].explode().unique()
m_filtered = m_codes[m_codes["SKSkode"].isin(all_m_codes)]

# Build hierarchy
m_hierarchy = SNOMEDHierarchy(m_filtered, main_len=2)

print("Number of available m codes: ", len(m_codes))
print("Number of m codes present in data set: ", len(m_filtered))

Number of available m codes:  7834
Number of m codes present in data set:  761


In [6]:
# Create topography hierarchy

# Get df of T codes
t_codes = snomed.get_codes_by_letter('T')

all_t_codes = df_selected["T"].explode().unique()
t_filtered = t_codes[t_codes["SKSkode"].isin(all_t_codes)]

# Build hierarchy
t_hierarchy = SNOMEDHierarchy(t_filtered, main_len=3)

print("Number of available t codes: ", len(t_codes))
print("Number of t codes present in data set: ", len(t_filtered))

Number of available t codes:  2289
Number of t codes present in data set:  464


In [7]:
t_hierarchy.print_all_regions(edited=False)

Region:  T00 Topografi ukendt

T000: Topografi ukendt (3 codes)
    T00001: Topografi ukendt
    T00002: Topografi kan ikke anvendes
    T00003: Topografi ikke anvendt

T001: Resektionslinie (2 codes)
    T00100: Resektionslinie
    T00100: Resektionsrand

----------------------------------------
Region:  T01 Hud

T010: Hud (1 codes)
    T01000: Hud

T012: Blodkar i hud (1 codes)
    T01230: Blodkar i hud

T013: Talgkirtel (2 codes)
    T01310: Talgkirtel
    T01320: Apokrin svedkirtel

T015: Øjenbryn (1 codes)
    T01520: Øjenbryn

T016: Negl (2 codes)
    T01600: Negl
    T01609: Negleleje

----------------------------------------
Region:  T02 Hud på hoved

T021: Hud på hoved (18 codes)
    T02100: Hud på hoved
    T02102: Hud på skalp
    T02104: Hud på pande
    T02105: Hud i supraorbitale region
    T02111: Hud i tinding
    T02113: Hud i postaurikulærregion
    T02114: Hud i præaurikulærregion
    T02120: Hud i ansigt
    T02121: Hud på kind
    T02130: Hud på øjenlåg
    T02131:

In [8]:
# Manually edit T hierarchy

# Update region name
t_hierarchy.update_region('T00', 'Uspecifik topografi')

# Merge regions T01, T02 and T03 into one main region T01+
t_hierarchy.merge_main_regions('T01+', ['T01', 'T02', 'T03'], new_name="Hud inkl. subcutis")

# Split T1X into T1X0 (Bløddelsvæv) and T1X+ (Knogle- og bruskvæv)
t_hierarchy.split_main_region('T1X', {'T1X0': ['T1X0'], 'T1X+': ['T1X5','T1X7']})

# Other updates
t_hierarchy.update_region('T04', 'Mammae')
t_hierarchy.merge_main_regions('T08+', ['T08', 'T09'], new_name="Lymfeknude og lymfekar")
t_hierarchy.merge_main_regions('T10+', ['T10', 'T11', 'T1X+'], new_name="Knogle")
t_hierarchy.update_region('T24', 'Larynx')
t_hierarchy.split_main_region('T2Y', {'T2Y4': ['T2Y4'], 'T2Y6': ['T2Y6']})
t_hierarchy.merge_main_regions('T26+', ['T26', 'T2Y4'], new_name="Bronchus")
t_hierarchy.merge_main_regions('T29+', ['T29', 'T2Y6'], new_name="Pleura")
t_hierarchy.merge_main_regions('T40+', ['T40', 'T45', 'T46', 'T48'], new_name="Blodkar")
t_hierarchy.update_region('T67', 'Colon')
t_hierarchy.update_region('T68', 'Rectum')
t_hierarchy.merge_main_regions('T71+', ['T71', 'T72'], new_name="Nyrer")
t_hierarchy.merge_main_regions('T74', ['T74', 'T7X'], new_name="Urinblære")
t_hierarchy.update_region('T79', 'Øvrige hanlige kønsorganer')
t_hierarchy.update_region('T93', 'Binyre')
t_hierarchy.update_region('TX2', 'Hjerne')
t_hierarchy.update_region('TXX', 'Øje')
t_hierarchy.merge_main_regions('T83+', ['T83','T8X'], new_name="Cervix Uteri")
t_hierarchy.merge_main_regions('T88+', ['T88','T89'])
# CONTINUE FROM T8X
# CONSIDERATIONS: broader groups, see below
"""
1. General / Undefined
T00 Topografi ukendt

2. Integumentary System (Hud og subcutis)
Hud (T01)
Hud efter region (T02)
Subcutis (T03)
Mamma (T04)

3. Hematopoietic & Lymphoid
Knoglemarv (T06)
Milt (T07)
Lymfeknuder (T08)
Lymfekar (T09)

4. Skelet & Bevægelsesapparat
Knogle (T10–T11)
Led (T12)
Muskler, sener og støttevæv
Skeletmuskulatur (T13)
Bursa (T16)
Sene (T17)
Ligament og fascie (T18)
Bløddelstypologi (T1X)

5. Luftveje
Næse & bihuler (T21–T22)
Pharynx / svælg (T23, T60–T63)
Tonsiller, adenoid og tilhørende væv (T61–T613)
Larynx & stemmebånd (T24)
Trachea & bronkier (T25–T26)
Lunger & pleura (T28–T29)
Cytologi luftveje (T2Y)

6. Hjerte & Kar
Endocardium (T34)
Blodkar (T40-T45-T46)
Vener (T48)

7. Fordøjelsessystemet
Mundhule (T51–T55)
Lever, galde, pancreas (T56–T59)
Mave-tarmkanalen (T63-T64-T65–T69)

8. Urinveje
Nyrer (T71)
Nyrepelvis (T72)
Ureter (T73)
Blære (T74)
Urethra (T75)
Cytologi urinveje (T7X)

9. Mandlige genitalia
Penis (T76)
Prostata & vesicula seminalis (T77)
Testis, epididymis, ductus deferens, funiculus spermaticus, scrotum (T78–T79)

10. Kvindelige genitalia
Vulva, labia, clitoris, Bartholin (T80)
Vagina (T81)
Uterus & cervix (T82–T83)
Endometrium, myometrium (T84–T85)
Tuba uterina & ovarier (T86–T87)
Placenta, fosterhinder, navlestreng (T88)
Foster (T89)
Cytologi cervix (T8X)

11. Endokrine kirtler
Binyre (T93)
Thyroidea & parathyroidea (T96–T97)
Thymus (T98)

12. Nervesystem & Sanseorganer
Hjerne (temporallap) (TX2)
Nerver (TX9)
Øjenlåg / conjunctiva (TXX)
Øre (ydre, mellemøre, øregang) (TXY)

13. Regionbaserede strukturer (TY-serie)
Hoved & hals (TY0)
Truncus (ryg, thorax, abdomen, pelvis, inguen) (TY1–TY7)
Ekstremiteter (over- og underekstremitet) (TY8–TY9)
"""

'\n1. General / Undefined\nT00 Topografi ukendt\n\n2. Integumentary System (Hud og subcutis)\nHud (T01)\nHud efter region (T02)\nSubcutis (T03)\nMamma (T04)\n\n3. Hematopoietic & Lymphoid\nKnoglemarv (T06)\nMilt (T07)\nLymfeknuder (T08)\nLymfekar (T09)\n\n4. Skelet & Bevægelsesapparat\nKnogle (T10–T11)\nLed (T12)\nMuskler, sener og støttevæv\nSkeletmuskulatur (T13)\nBursa (T16)\nSene (T17)\nLigament og fascie (T18)\nBløddelstypologi (T1X)\n\n5. Luftveje\nNæse & bihuler (T21–T22)\nPharynx / svælg (T23, T60–T63)\nTonsiller, adenoid og tilhørende væv (T61–T613)\nLarynx & stemmebånd (T24)\nTrachea & bronkier (T25–T26)\nLunger & pleura (T28–T29)\nCytologi luftveje (T2Y)\n\n6. Hjerte & Kar\nEndocardium (T34)\nBlodkar (T40-T45-T46)\nVener (T48)\n\n7. Fordøjelsessystemet\nMundhule (T51–T55)\nLever, galde, pancreas (T56–T59)\nMave-tarmkanalen (T63-T64-T65–T69)\n\n8. Urinveje\nNyrer (T71)\nNyrepelvis (T72)\nUreter (T73)\nBlære (T74)\nUrethra (T75)\nCytologi urinveje (T7X)\n\n9. Mandlige genitali

In [9]:
t_hierarchy.list_main_regions(edited = True)

T00: Uspecifik topografi (5)
T01+: Hud inkl. subcutis (72)
T04: Mammae (4)
T06: Knoglemarv (1)
T07: Milt (1)
T08+: Lymfeknude og lymfekar (29)
T10+: Knogle (16)
T12: Led (11)
T13: Skeletmuskulatur (1)
T16: Bursa (1)
T17: Sene (2)
T18: Ligament (4)
T1X0: Bløddelsvæv (2)
T21: Næse (6)
T22: Bihule (2)
T23: Næsesvælgrum (2)
T24: Larynx (9)
T25: Trachea (3)
T26+: Bronchus (12)
T28: Lunge (10)
T29+: Pleura (5)
T34: Endocardium (1)
T40+: Blodkar (4)
T51: Mund (9)
T52: Læbe (6)
T53: Tunge (4)
T54: Tand (3)
T55: Spytkirtel (4)
T56: Lever (2)
T57: Galdeblære (1)
T59: Pancreas (1)
T60: Pharynx (8)
T61: Tonsil og adenoid (11)
T62: Esophagus (2)
T63: Ventrikel (3)
T64: Tyndtarm (3)
T65: Ileum (3)
T66: Appendix (1)
T67: Colon (4)
T68: Rectum (1)
T69: Analkanal (1)
T71+: Nyrer (4)
T73: Ureter (3)
T75: Urethra (5)
T76: Penis (3)
T77: Prostata (6)
T78: Testis (8)
T79: Øvrige hanlige kønsorganer (13)
T80: Vulva, labia, clitoris og Bartholins kirtel (12)
T81: Vagina (3)
T82: Uterus (6)
T83+: Cervix Uteri

In [10]:
m_hierarchy.print_all_regions(edited=False)

Region:  M0 morfologiakse ikke anvendelig

M000: morfologiakse ikke anvendelig (1 codes)
    M00020: morfologiakse ikke anvendelig

M001: normalt væv (3 codes)
    M00100: normalt væv
    M00120: normale celler
    M00121: normale celler, ingen endocervikale el. metaplastiske celler

M004: normal cellularitet (1 codes)
    M00410: normal cellularitet

M010: abnormt væv (3 codes)
    M01000: abnormt væv
    M01040: abnorm struktur
    M01090: atypisk histologisk forandring

M011: uspecifik reaktiv forandring (1 codes)
    M01111: uspecifik reaktiv forandring

M025: abnormt lille størrelse (1 codes)
    M02520: abnormt lille størrelse

M028: abnorm høj vægt (2 codes)
    M02820: abnorm høj vægt
    M02840: abnorm lav vægt

M090: for lidt materiale til diagnostisk vurdering (12 codes)
    M09000: for lidt materiale til diagnostisk vurdering
    M09002: for lidt materiale til molekylærpatologisk undersøgelse
    M09010: materialet uegnet til diagnostisk vurdering
    M09011: materialet min

In [11]:
# Manually edit M hierarchy
m_hierarchy.update_region('M0', 'Unspecific morphology')
m_hierarchy.update_region('M1', 'Traumatic changes')
m_hierarchy.update_region('M2', 'Congenital malformations, pregnancy products')
m_hierarchy.update_region('M3', 'Mechanical changes')
m_hierarchy.update_region('M4', 'Inflammation and fibrosis')
m_hierarchy.update_region('M5', 'Degeneration, necrosis, deposition, dystrophy, atrophy') 
m_hierarchy.update_region('M6', 'Cellular changes') 
m_hierarchy.update_region('M7', 'Growth and maturation changes') 
m_hierarchy.merge_main_regions('M8-9', ['M8', 'M9'], new_name="Neoplasms")
m_hierarchy.update_region('MÆ', 'Description in text')

In [12]:
m_hierarchy.list_main_regions(edited = True)

M0: Unspecific morphology (47)
M1: Traumatic changes (16)
M2: Congenital malformations, pregnancy products (28)
M3: Mechanical changes (53)
M4: Inflammation and fibrosis (111)
M5: Degeneration, necrosis, deposition, dystrophy, atrophy (47)
M6: Cellular changes (9)
M7: Growth and maturation changes (116)
M8-9: Neoplasms (330)
MÆ: Description in text (4)


In [13]:
def map_codes_to_category(code_list, hierarchy):
    categories = []
    for code in code_list:
        region_name = hierarchy.code_to_main_region_name(code)
        categories.append(region_name)
    return list(set(categories))

In [14]:
df_selected["T category"] = df_selected["T"].apply(lambda x: map_codes_to_category(x, t_hierarchy))
df_selected["M category"] = df_selected["M"].apply(lambda x: map_codes_to_category(x, m_hierarchy))


Code T7X412 not found in code_to_region mapping.
In DataFrame: True
In code_to_region: False
In edited hierarchy: False
In original hierarchy: True
----------------------------------------
Code T74000 not found in code_to_region mapping.
In DataFrame: True
In code_to_region: False
In edited hierarchy: False
In original hierarchy: True
----------------------------------------
Code T74030 not found in code_to_region mapping.
In DataFrame: True
In code_to_region: False
In edited hierarchy: False
In original hierarchy: True
----------------------------------------
Code T74010 not found in code_to_region mapping.
In DataFrame: True
In code_to_region: False
In edited hierarchy: False
In original hierarchy: True
----------------------------------------
Code T74000 not found in code_to_region mapping.
In DataFrame: True
In code_to_region: False
In edited hierarchy: False
In original hierarchy: True
----------------------------------------
Code T74000 not found in code_to_region mapping.
In Dat

In [15]:
print(df_selected.head())


     rekvnr   modtdato team sex  alder alder gruppe   rekvdato  matantal  \
0  12100029 2012-01-02  LUG   F     87          65+ 2012-01-02         3   
1  12102211 2012-02-09  LUG   F     36        35-39 2012-02-07         1   
2  12105184 2012-03-28  LUG   M     87          65+ 2012-03-27         1   
3  12200001 2012-01-02  URO   M     52        50-54 2011-12-30        12   
4  12200014 2012-01-02  LUG   F     87          65+ 2012-01-02         1   

   mattype                                      wsi filenames  \
0       24  [//regsj.intern/appl/Deep_Visual_Proteomics\Sl...   
1       23  [//regsj.intern/appl/Deep_Visual_Proteomics\Sl...   
2       26  [//regsj.intern/appl/Deep_Visual_Proteomics\Sl...   
3       11  [//regsj.intern/appl/Deep_Visual_Proteomics\Sl...   
4       11  [//regsj.intern/appl/Deep_Visual_Proteomics\Sl...   

                          T                 M             Other  \
0  (T2Y414, T08B07, T08B44)  (M09462, M81406)  (P31067, ÆF4100)   
1                 